In [2]:
import ast
import time
import uuid
from pathlib import Path 

import cv2
import pandas as pd
from tqdm.auto import tqdm
from deepface import DeepFace
from qdrant_client import QdrantClient 
from qdrant_client.models import Distance, VectorParams, PointStruct

2025-01-26 17:36:35.394186: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
CLIENT = QdrantClient(host='192.168.0.131', port=6333)   

In [4]:
df = pd.read_csv('../data/recognition_model_comparison.csv', index_col=0)
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,mouth_right_y,mouth_left_x,mouth_left_y,confidence,face_num,frame_num,img_width,img_height,filepath,label
0,0.299,0.066,0.659,0.893,0.401,0.333,0.572,0.356,0.485,0.503,...,0.670,0.546,0.689,1.000,0,4224,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,Hugh Laurie
1,0.391,0.129,0.748,0.957,0.571,0.410,0.714,0.471,0.680,0.595,...,0.727,0.671,0.778,1.000,0,5304,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,Robert Sean Leonard
2,0.181,0.220,0.292,0.486,0.215,0.343,0.265,0.320,0.251,0.386,...,0.429,0.273,0.409,1.000,0,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,Robert Sean Leonard
3,0.529,0.152,0.656,0.454,0.546,0.283,0.595,0.273,0.561,0.341,...,0.391,0.601,0.383,0.998,1,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,Hugh Laurie
4,0.647,0.199,0.757,0.491,0.682,0.315,0.734,0.301,0.718,0.365,...,0.425,0.737,0.412,0.999,0,5664,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,Hugh Laurie


In [5]:
models = [
  "VGG-Face", 
  "Facenet", 
  "Facenet512", 
  "OpenFace", 
  # "DeepFace", 
  "DeepID", 
  "ArcFace", 
  "Dlib", 
  "SFace",
  "GhostFaceNet"
]

In [6]:
normalization = {
  "VGG-Face": "VGGFace2", 
  "Facenet": "Facenet", 
  "Facenet512": "Facenet", 
  "OpenFace": "base", 
  # "DeepFace", 
  "DeepID": "base", 
  "ArcFace": "ArcFace", 
  "Dlib": "base", 
  "SFace": "base",
  "GhostFaceNet": "base"
}

In [8]:
times = {}

In [9]:
cap = cv2.VideoCapture(df.at[0, 'filepath'])
for model in tqdm(models):
    name = f'encoding_{model}'
    if name in df.columns:
      if df[df[name].notna()].shape[0] == df.shape[0]:
        continue
    else:
      df[name] = None
    start = time.time()
    for frame_num in tqdm(df['frame_num'].unique().tolist(), leave=False):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        temp = df[df['frame_num'] == frame_num]
        for idx, row in temp.iterrows():
            x1 = int(row['x1'] * row['img_width'])
            y1 = int(row['y1'] * row['img_height'])
            x2 = int(row['x2'] * row['img_width'])
            y2 = int(row['y2'] * row['img_height'])
            face = frame[y1:y2, x1:x2]
            encoding = DeepFace.represent(face, 
                                          model_name=model,
                                          enforce_detection=False,
                                          detector_backend='skip',
                                          align=True,
                                          normalization=normalization[model],
                                          max_faces=1)
            df.at[idx, name] = encoding[0]['embedding']
            d = time.time() - start 
            times[model] = d

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/283 [00:00<?, ?it/s]

  0%|          | 0/283 [00:00<?, ?it/s]

  0%|          | 0/283 [00:00<?, ?it/s]

  0%|          | 0/283 [00:00<?, ?it/s]

  0%|          | 0/283 [00:00<?, ?it/s]

  0%|          | 0/283 [00:00<?, ?it/s]

  0%|          | 0/283 [00:00<?, ?it/s]

  0%|          | 0/283 [00:00<?, ?it/s]

  0%|          | 0/283 [00:00<?, ?it/s]

In [10]:
times

{'VGG-Face': 66.7795581817627,
 'Facenet': 109.29969429969788,
 'Facenet512': 109.82886934280396,
 'OpenFace': 81.62501502037048,
 'DeepID': 63.95687532424927,
 'ArcFace': 83.84205102920532,
 'Dlib': 89.7361330986023,
 'SFace': 65.5428159236908,
 'GhostFaceNet': 104.80306792259216}

In [11]:
df.to_csv('../data/recognition_model_comparison_encoded.csv')

In [12]:
headshot_dir = Path('../../data/headshots')
for model in tqdm(models):   
    headshots = [x for x in headshot_dir.iterdir()]
    for headshot in tqdm(headshots, leave=False):
        name, id_, num = headshot.stem.split('_')
        img = cv2.imread(str(headshot))
        encoding = DeepFace.represent(img,
                                      model_name=model,
                                      enforce_detection=True,
                                      detector_backend='retinaface',
                                      align=True,
                                      normalization=normalization[model],
                                      max_faces=1
                                      )
        collection_name = f'Headshots_{model}'
        collections = [x.name for x in CLIENT.get_collections().collections]
        if collection_name not in collections:
            CLIENT.recreate_collection(collection_name=collection_name,
                                    vectors_config=VectorParams(size=len(encoding[0]['embedding']), distance=Distance.COSINE))
        point = PointStruct(id=str(uuid.uuid4()),
                                   payload={
                                    'name': name,
                                    'tmdb_id': id_
                                   },
                                   vector=encoding[0]['embedding'])
        CLIENT.upsert(collection_name=collection_name,
                      points=[point]
                      )
        
    

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/38 [00:00<?, ?it/s]

In [13]:
for model in models:
    col = f'predicted_{model}' 
    encoding_col = f'encoding_{model}'
    collection_name = f'Headshots_{model}'
    if col not in df.columns:
        df[col] = None
    for idx, row in df.iterrows():
        # encoding = ast.literal_eval(row[encoding_col])
        encoding = row[encoding_col]
        response = CLIENT.query_points(collection_name=collection_name,
                                       query=encoding,
                                       limit=1)
        df.at[idx, col] = response.points[0].payload['name']

In [14]:
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,encoding_GhostFaceNet,predicted_VGG-Face,predicted_Facenet,predicted_Facenet512,predicted_OpenFace,predicted_DeepID,predicted_ArcFace,predicted_Dlib,predicted_SFace,predicted_GhostFaceNet
0,0.299,0.066,0.659,0.893,0.401,0.333,0.572,0.356,0.485,0.503,...,"[1.6946598291397095, -2.4882824420928955, 2.34...",Omar Epps,Hugh Laurie,Omar Epps,Hugh Laurie,Robert Sean Leonard,Hugh Laurie,Robert Sean Leonard,Andrew Airlie,Omar Epps
1,0.391,0.129,0.748,0.957,0.571,0.410,0.714,0.471,0.680,0.595,...,"[-0.608773946762085, -1.1071842908859253, -0.2...",Robert Sean Leonard,Robert Sean Leonard,Robert Sean Leonard,Jennifer Morrison,Omar Epps,Robert Sean Leonard,Robert Sean Leonard,Lisa Edelstein,Omar Epps
2,0.181,0.220,0.292,0.486,0.215,0.343,0.265,0.320,0.251,0.386,...,"[-2.170797824859619, 1.788697361946106, -1.553...",Robert Sean Leonard,Robert Sean Leonard,Robert Sean Leonard,Hugh Laurie,Omar Epps,Robert Sean Leonard,Ava Hughes,Robert Sean Leonard,Robert Sean Leonard
3,0.529,0.152,0.656,0.454,0.546,0.283,0.595,0.273,0.561,0.341,...,"[0.3280302584171295, -0.9947400093078613, -1.0...",Hugh Laurie,Hugh Laurie,Hugh Laurie,Hugh Laurie,Robert Sean Leonard,Hugh Laurie,Hugh Laurie,Omar Epps,Dylan Basu
4,0.647,0.199,0.757,0.491,0.682,0.315,0.734,0.301,0.718,0.365,...,"[-2.1556036472320557, -0.08507058024406433, 0....",Hugh Laurie,Hugh Laurie,Jesse Spencer,Jesse Spencer,Hugh Laurie,Hugh Laurie,Hugh Laurie,Hugh Laurie,Hugh Laurie


In [15]:
accuracy = {}
for model in models:
    col = f'predicted_{model}'
    results = df.apply(lambda x: 1 if x['label'] == x[col] else 0, axis=1)
    accuracy[model] = results.mean()

In [16]:
accuracy

{'VGG-Face': 0.8604651162790697,
 'Facenet': 0.936046511627907,
 'Facenet512': 0.7906976744186046,
 'OpenFace': 0.375,
 'DeepID': 0.09011627906976744,
 'ArcFace': 0.8691860465116279,
 'Dlib': 0.6366279069767442,
 'SFace': 0.5232558139534884,
 'GhostFaceNet': 0.5843023255813954}

In [79]:
result_df = pd.DataFrame.from_dict({k: {'time': times[k], 'accuracy': accuracy[k]} for k in accuracy.keys()})
result_df.transpose().sort_values(by='accuracy', ascending=False)

,time,accuracy
Facenet,107.715255,0.936047
ArcFace,83.249530,0.869186
VGG-Face,68.077897,0.860465
Facenet512,108.392452,0.790698
Dlib,90.204203,0.636628
GhostFaceNet,102.223972,0.584302
SFace,66.258777,0.523256
OpenFace,80.058715,0.375000
DeepID,64.372185,0.090116


In [76]:
old_accuracy = {'VGG-Face': 0.8604651162790697,
 'Facenet': 0.936046511627907,
 'Facenet512': 0.9069767441860465,
 'OpenFace': 0.375,
 'DeepID': 0.09011627906976744,
 'ArcFace': 0.8691860465116279,
 'Dlib': 0.6366279069767442,
 'SFace': 0.5232558139534884,
 'GhostFaceNet': 0.5843023255813954}

In [ ]:
df = 